# COVID-19 Data Analysis Project
## SoftGrowTech Data Science Internship
### Intern: Tahir Mahmood | Date: March 2026

## Cell 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

os.makedirs('images', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("=" * 70)
print("COVID-19 DATA ANALYSIS PROJECT")
print("=" * 70)
print("Libraries imported successfully!")
print("=" * 70)

## Cell 2: Load Dataset

In [ ]:
print("\n📁 STEP 1: Loading Dataset...")
print("-" * 50)

try:
    df = pd.read_csv("owid-covid-data.csv")
    print("✅ Dataset loaded from local file: owid-covid-data.csv")
except FileNotFoundError:
    print("⚠️ Local file not found. Downloading from URL...")
    url = "https://covid.ourworldindata.org/data/owid-covid-data.csv"
    df = pd.read_csv(url)
    print("✅ Dataset downloaded and loaded from Our World in Data")

print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
print(df.head())
print("-" * 50)

## Cell 3: Basic Dataset Information

In [ ]:
print("\n📊 STEP 2: Dataset Information...")
print("-" * 50)

print("\nDataset Info:")
print(df.info())

print("\nBasic Statistics:")
print(df.describe())

print("\nColumn Names:")
print(df.columns.tolist())
print("-" * 50)

## Cell 4: Missing Values Analysis

In [ ]:
print("\n🔍 STEP 3: Missing Values Analysis...")
print("-" * 50)

missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Percentage': missing_percentage
}).sort_values('Percentage', ascending=False)

print("Top 20 columns with missing values:")
print(missing_df[missing_df['Missing_Count'] > 0].head(20))
print("-" * 50)

## Cell 5: Data Cleaning & Preprocessing

In [ ]:
print("\n🧹 STEP 4: Data Cleaning & Preprocessing...")
print("-" * 50)

important_cols = [
    'location', 'date', 'total_cases', 'new_cases',
    'total_deaths', 'new_deaths', 'total_vaccinations',
    'people_vaccinated', 'people_fully_vaccinated',
    'population', 'continent'
]

available_cols = [col for col in important_cols if col in df.columns]
df_clean = df[available_cols].copy()

df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.fillna(0)
df_clean = df_clean.sort_values(['location', 'date'])

print(f"✅ Cleaned Dataset Info:")
print(f"   - Rows: {df_clean.shape[0]:,}")
print(f"   - Columns: {df_clean.shape[1]}")
print(f"   - Date Range: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")
print("\nFirst 5 rows of cleaned data:")
print(df_clean.head())
print("-" * 50)

## Cell 6: Global Overview - Latest Statistics

In [ ]:
print("\n🌍 STEP 5: Global Overview - Latest Statistics...")
print("-" * 50)

latest_date = df_clean['date'].max()
latest_data = df_clean[df_clean['date'] == latest_date].copy()

latest_data = latest_data[
    (~latest_data['location'].isin(['World', 'International'])) & 
    (latest_data['continent'] != '')
]

top_countries_cases = latest_data.nlargest(20, 'total_cases')[['location', 'total_cases', 'total_deaths', 'population']]

print(f"📅 Latest Data Date: {latest_date.date()}")
print("\n🏆 Top 20 Countries by Total Cases:")
print(top_countries_cases.to_string(index=False))
print("-" * 50)

## Cell 7: Visualization 1 - Top 10 Countries by Total Cases

In [ ]:
print("\n📈 STEP 6: Creating Visualizations...")
print("-" * 50)
print("1️⃣ Top 10 Countries by Total Cases")

plt.figure(figsize=(14, 8))

top10_cases = top_countries_cases.head(10)
colors = plt.cm.Reds(np.linspace(0.4, 0.9, 10))

bars = plt.barh(top10_cases['location'], top10_cases['total_cases'], color=colors)
plt.xlabel('Total Cases (in millions)', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.title('Top 10 Countries by Total COVID-19 Cases', fontsize=16, fontweight='bold')

for i, (bar, value) in enumerate(zip(bars, top10_cases['total_cases'])):
    plt.text(value, bar.get_y() + bar.get_height()/2, 
             f'{value/1e6:.1f}M', 
             ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('images/top_countries.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/top_countries.png")

## Cell 8: Visualization 2 - Cases vs Deaths Scatter Plot

In [ ]:
print("\n2️⃣ Cases vs Deaths Scatter Plot")

plt.figure(figsize=(12, 8))

latest_data['death_rate'] = (latest_data['total_deaths'] / latest_data['total_cases']) * 100

scatter = plt.scatter(latest_data['total_cases']/1e6, 
                     latest_data['total_deaths']/1e3,
                     c=latest_data['death_rate'],
                     s=latest_data['population']/1e6,
                     alpha=0.6,
                     cmap='RdYlGn_r')

plt.colorbar(scatter, label='Death Rate (%)')
plt.xlabel('Total Cases (Millions)', fontsize=12)
plt.ylabel('Total Deaths (Thousands)', fontsize=12)
plt.title('COVID-19: Total Cases vs Deaths by Country', fontsize=16, fontweight='bold')

major_countries = ['United States', 'India', 'Brazil', 'Russia', 'United Kingdom']
for country in major_countries:
    country_data = latest_data[latest_data['location'] == country]
    if not country_data.empty:
        plt.annotate(country, 
                    (country_data['total_cases'].values[0]/1e6, 
                     country_data['total_deaths'].values[0]/1e3),
                    fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('images/scatter_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/scatter_plot.png")

## Cell 9: Time Series Data Preparation

In [ ]:
print("\n3️⃣ Time Series Analysis")

selected_countries = ['United States', 'India', 'Brazil', 'United Kingdom', 'Japan', 'South Africa']
ts_data = df_clean[df_clean['location'].isin(selected_countries)]

print(f"Selected Countries: {selected_countries}")
print(f"Date Range: {ts_data['date'].min().date()} to {ts_data['date'].max().date()}")
print(f"Total Records: {len(ts_data):,}")

## Cell 10: Visualization 3 - Daily New Cases Over Time

In [ ]:
plt.figure(figsize=(16, 8))

for country in selected_countries:
    country_data = ts_data[ts_data['location'] == country]
    plt.plot(country_data['date'], country_data['new_cases'], 
             label=country, linewidth=2)

plt.xlabel('Date', fontsize=12)
plt.ylabel('Daily New Cases', fontsize=12)
plt.title('Daily New COVID-19 Cases Over Time', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x/1e6), '') + 'M'))

plt.tight_layout()
plt.savefig('images/time_series.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/time_series.png")

## Cell 11: Visualization 4 - Cumulative Cases Over Time

In [ ]:
plt.figure(figsize=(16, 8))

for country in selected_countries:
    country_data = ts_data[ts_data['location'] == country]
    plt.plot(country_data['date'], country_data['total_cases']/1e6, 
             label=country, linewidth=2)

plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Cases (Millions)', fontsize=12)
plt.title('Cumulative COVID-19 Cases Over Time', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/cumulative_cases.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/cumulative_cases.png")

## Cell 12: Visualization 5 - Heatmap of COVID-19 Impact

In [ ]:
print("\n4️⃣ Heatmap Analysis")

monthly_data = df_clean.copy()
monthly_data['year_month'] = monthly_data['date'].dt.to_period('M')

top15_countries = latest_data.nlargest(15, 'total_cases')['location'].tolist()

heatmap_data = monthly_data[monthly_data['location'].isin(top15_countries)]
heatmap_pivot = heatmap_data.pivot_table(
    values='new_cases', 
    index='location', 
    columns='year_month', 
    aggfunc='sum'
)

heatmap_pivot = heatmap_pivot.fillna(0)

plt.figure(figsize=(20, 10))
sns.heatmap(heatmap_pivot/1e6, cmap='YlOrRd', annot=False, 
            cbar_kws={'label': 'Monthly New Cases (Millions)'})
plt.title('Monthly New COVID-19 Cases Heatmap - Top 15 Countries', fontsize=16, fontweight='bold')
plt.xlabel('Year-Month', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('images/heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/heatmap.png")

## Cell 13: Vaccination Analysis

In [ ]:
print("\n💉 STEP 7: Vaccination Analysis...")
print("-" * 50)

if 'total_vaccinations' in df_clean.columns:
    latest_vacc = df_clean[df_clean['date'] == latest_date].copy()
    latest_vacc = latest_vacc[latest_vacc['total_vaccinations'] > 0]
    latest_vacc = latest_vacc.nlargest(15, 'total_vaccinations')
    
    latest_vacc['vaccination_rate'] = (latest_vacc['people_vaccinated'] / latest_vacc['population']) * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    axes[0].barh(latest_vacc['location'], latest_vacc['total_vaccinations']/1e6, color='skyblue')
    axes[0].set_xlabel('Total Vaccinations (Millions)')
    axes[0].set_title('Top 15 Countries by Total Vaccinations')
    
    axes[1].barh(latest_vacc['location'], latest_vacc['vaccination_rate'], color='lightgreen')
    axes[1].set_xlabel('Vaccination Rate (%)')
    axes[1].set_title('Vaccination Rate (% of Population)')
    
    plt.tight_layout()
    plt.savefig('images/vaccination_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nTop 10 Countries by Vaccination Rate:")
    print(latest_vacc[['location', 'vaccination_rate']].head(10).to_string(index=False))
    print("✅ Saved: images/vaccination_analysis.png")
else:
    print("⚠️ Vaccination data not available in this dataset")
print("-" * 50)

## Cell 14: Continent-wise Analysis

In [ ]:
print("\n🌎 STEP 8: Continent-wise Analysis...")
print("-" * 50)

continent_data = df_clean[df_clean['continent'].notna()].copy()
continent_latest = continent_data[continent_data['date'] == latest_date]

continent_summary = continent_latest.groupby('continent').agg({
    'total_cases': 'sum',
    'total_deaths': 'sum',
    'population': 'sum'
}).reset_index()

continent_summary['death_rate'] = (continent_summary['total_deaths'] / continent_summary['total_cases']) * 100
continent_summary['cases_per_million'] = (continent_summary['total_cases'] / continent_summary['population']) * 1e6

print("Continent-wise COVID-19 Statistics:")
print(continent_summary.to_string(index=False))
print("-" * 50)

## Cell 15: Visualization 6 - Continent Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].pie(continent_summary['total_cases'], labels=continent_summary['continent'],
               autopct='%1.1f%%', startangle=90, colors=plt.cm.Set3(range(6)))
axes[0, 0].set_title('Distribution of COVID-19 Cases by Continent', fontweight='bold')

x_pos = np.arange(len(continent_summary['continent']))
axes[0, 1].bar(x_pos, continent_summary['total_deaths']/1e6,
               color=['red', 'blue', 'green', 'orange', 'purple', 'brown'])
axes[0, 1].set_xlabel('Continent')
axes[0, 1].set_ylabel('Total Deaths (Millions)')
axes[0, 1].set_title('Total COVID-19 Deaths by Continent', fontweight='bold')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(continent_summary['continent'], rotation=45)

axes[1, 0].bar(x_pos, continent_summary['death_rate'], color='salmon')
axes[1, 0].set_xlabel('Continent')
axes[1, 0].set_ylabel('Death Rate (%)')
axes[1, 0].set_title('COVID-19 Death Rate by Continent', fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(continent_summary['continent'], rotation=45)

axes[1, 1].bar(x_pos, continent_summary['cases_per_million']/1000, color='lightblue')
axes[1, 1].set_xlabel('Continent')
axes[1, 1].set_ylabel('Cases per Million (Thousands)')
axes[1, 1].set_title('COVID-19 Cases per Million Population', fontweight='bold')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(continent_summary['continent'], rotation=45)

plt.tight_layout()
plt.savefig('images/continent_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/continent_comparison.png")

## Cell 16: Growth Rate Analysis

In [ ]:
print("\n📈 STEP 9: Growth Rate Analysis...")
print("-" * 50)

growth_analysis = ts_data.copy()
growth_analysis['growth_rate'] = growth_analysis.groupby('location')['new_cases'].pct_change() * 100

latest_30_days = growth_analysis[growth_analysis['date'] > (growth_analysis['date'].max() - pd.Timedelta(days=30))]
avg_growth = latest_30_days.groupby('location')['growth_rate'].mean().reset_index()
avg_growth = avg_growth.dropna()

print("Average Daily Growth Rate (Last 30 Days):")
for index, row in avg_growth.iterrows():
    print(f"{row['location']}: {row['growth_rate']:.2f}%")
print("-" * 50)

## Cell 17: Correlation Analysis

In [ ]:
print("\n🔗 STEP 10: Correlation Analysis...")
print("-" * 50)

corr_columns = ['total_cases', 'new_cases', 'total_deaths', 'new_deaths']
if 'total_vaccinations' in df_clean.columns:
    corr_columns.extend(['total_vaccinations', 'people_vaccinated'])

corr_data = latest_data[corr_columns].dropna()
correlation_matrix = corr_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - COVID-19 Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: images/correlation_matrix.png")

## Cell 18: Summary Statistics

In [ ]:
print("\n📊 STEP 11: Summary Statistics...")
print("=" * 70)
print("COVID-19 DATA ANALYSIS SUMMARY")
print("=" * 70)

global_total_cases = df_clean[df_clean['location'] == 'World']['total_cases'].max()
global_total_deaths = df_clean[df_clean['location'] == 'World']['total_deaths'].max()
global_death_rate = (global_total_deaths / global_total_cases) * 100

print(f"\nGLOBAL STATISTICS (as of {latest_date.date()}):")
print(f"Total Cases Worldwide: {global_total_cases:,.0f}")
print(f"Total Deaths Worldwide: {global_total_deaths:,.0f}")
print(f"Global Death Rate: {global_death_rate:.2f}%")

print(f"\nTOP 5 COUNTRIES BY TOTAL CASES:")
for idx, row in top_countries_cases.head(5).iterrows():
    death_rate = (row['total_deaths'] / row['total_cases']) * 100 if row['total_cases'] > 0 else 0
    print(f"{row['location']}: {row['total_cases']:,.0f} cases, {death_rate:.2f}% death rate")

print(f"\nCOUNTRIES WITH HIGHEST DEATH RATE (>5%):")
high_death_rate = latest_data[latest_data['total_cases'] > 100000].nlargest(10, 'death_rate')[['location', 'death_rate']]
for idx, row in high_death_rate.iterrows():
    print(f"{row['location']}: {row['death_rate']:.2f}%")

## Cell 19: Save Results

In [ ]:
print("\n💾 STEP 12: Saving Results...")
print("-" * 50)

output_file = 'outputs/analysis_results.csv'

summary_data = {
    'Metric': ['Total Global Cases', 'Total Global Deaths', 'Global Death Rate', 
               'Number of Countries Analyzed', 'Date Range Start', 'Date Range End',
               'Analysis Date', 'Intern Name', 'Institution'],
    'Value': [f"{global_total_cases:,.0f}", 
              f"{global_total_deaths:,.0f}", 
              f"{global_death_rate:.2f}%",
              f"{len(latest_data)}", 
              df_clean['date'].min().strftime('%Y-%m-%d'),
              df_clean['date'].max().strftime('%Y-%m-%d'),
              datetime.now().strftime('%Y-%m-%d'),
              'Tahir Mahmood',
              'SoftGrowTech']
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(output_file, index=False)

print(f"✅ Summary data saved to: {output_file}")
print("-" * 50)

## Cell 20: Interactive Visualization

In [ ]:
print("\n🌐 STEP 13: Creating Interactive Visualization...")
print("-" * 50)

try:
    fig = px.choropleth(latest_data, 
                        locations='location',
                        locationmode='country names',
                        color='total_cases',
                        hover_name='location',
                        hover_data={'total_cases': True, 'total_deaths': True, 'population': True},
                        color_continuous_scale='Reds',
                        title='World Map: Total COVID-19 Cases by Country',
                        labels={'total_cases': 'Total Cases'})
    
    fig.update_layout(title_font_size=20, title_x=0.5)
    fig.write_html("outputs/interactive_map.html")
    print("✅ Interactive map saved as: outputs/interactive_map.html")
    fig.show()
    
except Exception as e:
    print(f"⚠️ Interactive map could not be displayed: {e}")
print("-" * 50)

## Cell 21: Project Completion

In [ ]:
print("\n" + "=" * 70)
print("🚀 PROJECT COMPLETED SUCCESSFULLY! 🚀")
print("=" * 70)

print("\n📁 Generated Files:")
print("   📊 images/top_countries.png")
print("   📊 images/scatter_plot.png")
print("   📊 images/time_series.png")
print("   📊 images/cumulative_cases.png")
print("   📊 images/heatmap.png")
if 'total_vaccinations' in df_clean.columns:
    print("   📊 images/vaccination_analysis.png")
print("   📊 images/continent_comparison.png")
print("   📊 images/correlation_matrix.png")
print("   📄 outputs/analysis_results.csv")
print("   🌐 outputs/interactive_map.html")

print("\n📌 Intern: Tahir Mahmood")
print("🏢 Organization: SoftGrowTech")
print(f"📅 Date: March 2026")

print("\n" + "=" * 70)
print("🎯 Thank you for reviewing my project!")
print("=" * 70)